# Capítulo 6 — Software Timer Management (FreeRTOS)

**Libro:** Mastering the FreeRTOS Real Time Kernel v1.1.0 — Richard Barry  
**Hardware de referencia:** STM32 Nucleo F103RB  
**APIs clave:** `timers.h`, `FreeRTOSConfig.h`


## 6.1 Introducción — ¿Qué es un Software Timer?

Un **software timer** permite ejecutar una función (su *callback*) en un momento futuro o a intervalos periódicos. A diferencia de los timers hardware, los software timers **no usan ningún recurso hardware** y no requieren soporte especial del microcontrolador.

**Características clave:**
- Son puramente software: implementados en el kernel de FreeRTOS.
- El tiempo se mide en *ticks* del scheduler de FreeRTOS.
- No consumen CPU mientras no están ejecutando su callback.
- Deben habilitarse explícitamente en `FreeRTOSConfig.h`.

### Constantes de configuración en `FreeRTOSConfig.h`

```c
/* Habilitar soporte de software timers */
#define configUSE_TIMERS              1

/* Prioridad de la tarea daemon que ejecuta los callbacks.
   Debe ser mayor que 0. Recomendado: mayor que las tareas de aplicación
   para que los callbacks se ejecuten a tiempo. */
#define configTIMER_TASK_PRIORITY     (configMAX_PRIORITIES - 1)

/* Longitud de la cola de comandos del timer (timer command queue) */
#define configTIMER_QUEUE_LENGTH      10

/* Tamaño del stack de la tarea daemon del timer */
#define configTIMER_TASK_STACK_DEPTH  configMINIMAL_STACK_SIZE
```

**Importante:** Si `configUSE_TIMERS` es 0, ninguna API de `timers.h` está disponible.


## 6.2 Funciones Callback de un Software Timer

Cada software timer tiene asociada una **función callback** que se ejecuta cuando el timer expira. El prototipo es fijo:

```c
void ATimerCallback(TimerHandle_t xTimer);
```

El parámetro `xTimer` es el handle del timer que expiró, útil cuando un mismo callback es compartido por varios timers.

### Reglas obligatorias para callbacks

1. **Deben ejecutarse de principio a fin sin bloquearse.** No pueden llamar a funciones que pongan la tarea en estado Blocked.
2. **No llamar a `vTaskDelay()`** — siempre pone la tarea en Blocked.
3. **`xQueueReceive()` solo con `xTicksToWait = 0`** — con cualquier otro valor bloquea la tarea.
4. **Deben ser cortas.** Ejecutan en el contexto de la tarea daemon; si son lentas, retrasan otros timers.

```c
/* Ejemplo de callback correcto: */
void vMyTimerCallback(TimerHandle_t xTimer) {
    /* Operación rápida, sin bloqueo */
    HAL_GPIO_TogglePin(GPIOA, GPIO_PIN_5);  /* Toggle LED */
}

/* Ejemplo INCORRECTO — nunca hacer esto en un callback: */
void vBadCallback(TimerHandle_t xTimer) {
    vTaskDelay(pdMS_TO_TICKS(100));  /* ERROR: bloquea la tarea daemon */
}
```

> **Nota:** Los callbacks ejecutan en el contexto de la **tarea daemon** (también llamada *timer service task*), que FreeRTOS crea automáticamente al iniciar el scheduler. Por eso no pueden bloquear: bloquearían la daemon task, impidiendo que otros timers procesen sus callbacks.


## 6.3 Tipos y Estados de un Software Timer

### 6.3.1 Período

El **período** de un software timer es el tiempo entre que el timer es iniciado y que su callback se ejecuta. Se especifica en ticks; usar `pdMS_TO_TICKS()` para convertir desde milisegundos.

### 6.3.2 Tipos: One-shot vs Auto-reload

| Tipo | Comportamiento | `xAutoReload` |
|------|---------------|---------------|
| **One-shot** | El callback ejecuta **una sola vez**. El timer pasa a estado Dormant después. Puede reiniciarse manualmente. | `pdFALSE` |
| **Auto-reload** | El callback ejecuta **repetidamente** con el período definido. El timer se reinicia automáticamente. | `pdTRUE` |

```
Timer 1 (one-shot,  período 6):
t1─────────────────────t7 → callback ejecuta UNA VEZ → Dormant

Timer 2 (auto-reload, período 5):
t1────t6 → callback → t11 → callback → t16 → callback → ...
```

### 6.3.3 Estados: Dormant y Running

| Estado | Descripción |
|--------|-------------|
| **Dormant** | El timer existe (tiene handle) pero **no está contando**. Su callback no se ejecutará. |
| **Running** | El timer está contando. Su callback ejecutará cuando expire el período. |

**Transiciones de estado:**

```
xTimerCreate() ──→ [Dormant]

[Dormant] ──xTimerStart() / xTimerReset() / xTimerChangePeriod()──→ [Running]
[Running] ──xTimerStop()──→ [Dormant]

Auto-reload: [Running] ──timer expira / ejecuta callback──→ [Running]  (se reinicia)
One-shot:    [Running] ──timer expira / ejecuta callback──→ [Dormant]
```

Un timer recién creado siempre comienza en estado **Dormant**.


## 6.4 Contexto de Ejecución: La Tarea Daemon y la Timer Command Queue

### 6.4.1 La RTOS Daemon Task (Timer Service Task)

Todos los callbacks de software timers ejecutan en el contexto de la misma tarea: la **RTOS daemon task** (también llamada *timer service task*).

- Se crea automáticamente cuando el scheduler arranca (`vTaskStartScheduler()`).
- Su prioridad y stack se configuran con `configTIMER_TASK_PRIORITY` y `configTIMER_TASK_STACK_DEPTH`.
- Es una tarea FreeRTOS normal: el scheduler la ejecuta cuando es la tarea de mayor prioridad en estado Ready.

### 6.4.2 La Timer Command Queue

Las APIs de software timers (`xTimerStart`, `xTimerStop`, `xTimerReset`, etc.) **no actúan directamente** sobre el timer. En cambio, envían un **comando** a la *timer command queue*, una cola FreeRTOS estándar creada automáticamente al iniciar el scheduler.

```
Tarea de aplicación          Timer Command Queue         Daemon Task
────────────────────         ───────────────────         ───────────
xTimerStart(xTimer, 0) ──→  ["start timer" command] ──→ procesa comando
                                                          ejecuta callback cuando expira
```

La longitud de esta cola se configura con `configTIMER_QUEUE_LENGTH`. Si la cola está llena, el llamador puede quedar bloqueado esperando (según el `xTicksToWait` que se pase a la API).

**Los comandos incluyen un timestamp:** el tiempo se calcula desde cuando el comando fue enviado a la cola, no desde cuando la daemon task lo procesa. Esto garantiza que el timer expire en el tiempo correcto aunque haya demora en procesarse.

### 6.4.3 Impacto de la Prioridad de la Daemon Task

La daemon task se comporta como cualquier tarea FreeRTOS: solo ejecuta cuando es la de mayor prioridad en estado Ready.

**Caso 1 — Daemon task con prioridad MENOR que la tarea llamadora:**
```
t1: Task1 ejecuta (Running)
t2: Task1 llama xTimerStart() → comando enviado a la cola → Daemon pasa a Ready
    (Daemon NO pre-empta a Task1 porque tiene menor prioridad)
t3: Task1 termina xTimerStart() sin haber dejado el estado Running
t4: Task1 se bloquea → Daemon pasa a Running y procesa el comando
```

**Caso 2 — Daemon task con prioridad MAYOR que la tarea llamadora:**
```
t1: Task1 ejecuta (Running)
t2: Task1 llama xTimerStart() → comando enviado → Daemon pre-empta a Task1 inmediatamente
t3: Daemon procesa el comando → vuelve a Blocked
t4: Task1 retoma y completa xTimerStart()
```

> **Recomendación práctica:** Para STM32, `configTIMER_TASK_PRIORITY` igual a `(configMAX_PRIORITIES - 1)` garantiza que los callbacks se ejecuten tan pronto como el timer expira.


## 6.5 Crear e Iniciar un Software Timer

### 6.5.1 `xTimerCreate()`

```c
TimerHandle_t xTimerCreate(
    const char * const pcTimerName,       /* Nombre para depuración */
    const TickType_t   xTimerPeriodInTicks, /* Período en ticks */
    const BaseType_t   xAutoReload,        /* pdTRUE=auto-reload, pdFALSE=one-shot */
    void * const       pvTimerID,          /* ID inicial (void*, uso libre) */
    TimerCallbackFunction_t pxCallbackFunction /* Función callback */
);
```

**Parámetros:**
- `pcTimerName`: nombre de texto, solo para depuración. FreeRTOS no lo usa internamente.
- `xTimerPeriodInTicks`: período en ticks. Usar `pdMS_TO_TICKS(ms)` para convertir. No puede ser 0.
- `xAutoReload`: `pdTRUE` → auto-reload; `pdFALSE` → one-shot.
- `pvTimerID`: puntero `void*` de uso libre. Puede almacenar un entero, puntero a datos, etc.
- `pxCallbackFunction`: puntero a la función callback (simplemente el nombre de la función).

**Retorno:** `NULL` si no hay heap suficiente; handle válido (`TimerHandle_t`) si exitoso.

El timer se crea en estado **Dormant**. Nunca en estado Running.

### 6.5.2 `xTimerStart()`

```c
BaseType_t xTimerStart(TimerHandle_t xTimer, TickType_t xTicksToWait);
```

- Inicia un timer en Dormant, o **reinicia** uno en Running (equivale a reset).
- `xTicksToWait`: tiempo máximo a esperar si la timer command queue está llena.
- Si se llama **antes** de `vTaskStartScheduler()`, el valor de `xTicksToWait` se ignora.
- **Nunca llamar desde una ISR** — usar `xTimerStartFromISR()` en su lugar.
- Retorna `pdPASS` si el comando fue enviado a la cola; `pdFAIL` si la cola estaba llena.

### Ejemplo 6.1 — Crear y arrancar un timer one-shot y uno auto-reload

```c
#include "FreeRTOS.h"
#include "timers.h"

#define mainONE_SHOT_TIMER_PERIOD    pdMS_TO_TICKS(3333)
#define mainAUTO_RELOAD_TIMER_PERIOD pdMS_TO_TICKS(500)

static void prvOneShotTimerCallback(TimerHandle_t xTimer) {
    TickType_t xTimeNow = xTaskGetTickCount();
    /* Este callback solo se ejecuta UNA vez (3333 ms después de iniciar) */
    vPrintStringAndNumber("One-shot timer callback executing", xTimeNow);
}

static void prvAutoReloadTimerCallback(TimerHandle_t xTimer) {
    TickType_t xTimeNow = xTaskGetTickCount();
    /* Este callback se ejecuta cada 500 ms indefinidamente */
    vPrintStringAndNumber("Auto-reload timer callback executing", xTimeNow);
}

int main(void) {
    TimerHandle_t xOneShotTimer, xAutoReloadTimer;
    BaseType_t xTimer1Started, xTimer2Started;

    xOneShotTimer = xTimerCreate(
        "OneShot",
        mainONE_SHOT_TIMER_PERIOD,
        pdFALSE,                    /* one-shot */
        0,                          /* ID inicial = 0 */
        prvOneShotTimerCallback
    );

    xAutoReloadTimer = xTimerCreate(
        "AutoReload",
        mainAUTO_RELOAD_TIMER_PERIOD,
        pdTRUE,                     /* auto-reload */
        0,
        prvAutoReloadTimerCallback
    );

    if ((xOneShotTimer != NULL) && (xAutoReloadTimer != NULL)) {
        /* Antes del scheduler: xTicksToWait se ignora, usar 0 */
        xTimer1Started = xTimerStart(xOneShotTimer, 0);
        xTimer2Started = xTimerStart(xAutoReloadTimer, 0);

        if ((xTimer1Started == pdPASS) && (xTimer2Started == pdPASS)) {
            vTaskStartScheduler();
        }
    }
    for (;;);
}
```

**Salida esperada:**
```
Auto-reload timer callback executing  500
Auto-reload timer callback executing  1000
Auto-reload timer callback executing  1500
...
Auto-reload timer callback executing  3000
One-shot timer callback executing     3333   ← solo una vez
Auto-reload timer callback executing  3500
Auto-reload timer callback executing  4000
...
```


## 6.6 Timer ID — Identificación y Almacenamiento por Timer

Cada software timer tiene un **ID**: un puntero `void*` de uso libre, asignado al crear el timer con `pvTimerID` y actualizable en cualquier momento.

**Usos típicos del ID:**
- Distinguir qué timer expiró cuando un mismo callback es compartido por varios timers.
- Almacenar un contador de ejecuciones por timer (como `uint32_t` casteado a `void*`).
- Apuntar a datos de contexto específicos de cada timer.

```c
/* Establecer el ID de un timer */
void vTimerSetTimerID(const TimerHandle_t xTimer, void *pvNewID);

/* Leer el ID de un timer */
void *pvTimerGetTimerID(const TimerHandle_t xTimer);
```

Estas funciones acceden al timer **directamente** (no pasan por la timer command queue), por lo que son inmediatas.

### Ejemplo 6.2 — Un callback compartido que usa el ID como contador

```c
/* Un único callback para ambos timers: one-shot y auto-reload */
static void prvTimerCallback(TimerHandle_t xTimer) {
    TickType_t xTimeNow;
    uint32_t ulExecutionCount;

    /* El ID almacena el número de ejecuciones de ESTE timer */
    ulExecutionCount = (uint32_t) pvTimerGetTimerID(xTimer);
    ulExecutionCount++;
    vTimerSetTimerID(xTimer, (void *) ulExecutionCount);

    xTimeNow = xTaskGetTickCount();

    if (xTimer == xOneShotTimer) {
        vPrintStringAndNumber("One-shot timer callback executing", xTimeNow);
    } else {
        vPrintStringAndNumber("Auto-reload timer callback executing", xTimeNow);

        /* Detener el auto-reload timer al 5to disparo */
        if (ulExecutionCount == 5) {
            /* Callback ejecuta en la daemon task: usar xTicksToWait = 0
               para no bloquear la daemon task */
            xTimerStop(xTimer, 0);
        }
    }
}
```

**Creación con callback compartido:**
```c
xOneShotTimer = xTimerCreate("OneShot",
                              mainONE_SHOT_TIMER_PERIOD,
                              pdFALSE,
                              NULL,            /* ID inicia en NULL */
                              prvTimerCallback);

xAutoReloadTimer = xTimerCreate("AutoReload",
                                 mainAUTO_RELOAD_TIMER_PERIOD,
                                 pdTRUE,
                                 NULL,
                                 prvTimerCallback); /* mismo callback */
```

**Salida esperada:** el auto-reload timer ejecuta 5 veces (a 500, 1000, 1500, 2000, 2500 ms) y se detiene. El one-shot ejecuta una sola vez a 3333 ms.


## 6.7 Cambiar el Período de un Timer — `xTimerChangePeriod()`

```c
BaseType_t xTimerChangePeriod(
    TimerHandle_t xTimer,
    TickType_t    xNewPeriod,       /* Nuevo período en ticks */
    TickType_t    xTicksToWait      /* Tiempo máximo a esperar si la cola está llena */
);
```

**Comportamientos:**
- Si el timer está en estado **Running**: recalcula el tiempo de expiración desde el momento en que se llama (no desde cuando fue iniciado originalmente).
- Si el timer está en estado **Dormant**: calcula el tiempo de expiración y **pasa a Running automáticamente** (arranca el timer).
- **Nunca llamar desde ISR** — usar `xTimerChangePeriodFromISR()` en su lugar.
- Retorna `pdPASS` / `pdFAIL` según si el comando pudo enviarse a la cola.

### Caso de uso típico: LED indicador de estado

```c
/* El check timer togglea un LED cada 3 segundos si todo está bien.
   Si detecta un error, reduce el período a 200 ms (toggle rápido). */

const TickType_t xHealthyTimerPeriod = pdMS_TO_TICKS(3000);
const TickType_t xErrorTimerPeriod   = pdMS_TO_TICKS(200);

static void prvCheckTimerCallbackFunction(TimerHandle_t xTimer) {
    static BaseType_t xErrorDetected = pdFALSE;

    if (xErrorDetected == pdFALSE) {
        if (CheckTasksAreRunningWithoutError() == pdFAIL) {
            /* Error detectado: reducir el período para toggle rápido.
               xTicksToWait = 0 porque estamos en callback (daemon task). */
            xTimerChangePeriod(xTimer, xErrorTimerPeriod, 0);
            xErrorDetected = pdTRUE;
        }
    }
    ToggleLED();
}
```

> **Nota:** Al llamar `xTimerChangePeriod()` desde dentro de un callback, siempre usar `xTicksToWait = 0`. Si no, se bloquearía la daemon task.


## 6.8 Resetear un Software Timer — `xTimerReset()`

**Resetear** un timer significa reiniciar su cuenta: el tiempo de expiración se recalcula desde el momento del reset, no desde cuando fue iniciado originalmente.

```c
BaseType_t xTimerReset(TimerHandle_t xTimer, TickType_t xTicksToWait);
```

- Si el timer está en **Running**: reinicia el conteo desde el momento actual.
- Si el timer está en **Dormant**: equivale a `xTimerStart()` — pone el timer en Running.
- **Nunca llamar desde ISR** — usar `xTimerResetFromISR()` en su lugar.

**Comportamiento del reset (timer con período 6):**
```
t1: Timer iniciado → expiraría en t7
t5: Reset → expiraría en t11
t9: Reset → expiraría en t15
t15: Expira → callback ejecuta
```

### Ejemplo 6.3 — Simulación de backlight de celular

El comportamiento a simular:
- La pantalla se enciende al presionar una tecla.
- Permanece encendida si se siguen presionando teclas dentro del período.
- Se apaga automáticamente si no se presiona ninguna tecla durante el período.

**Implementación:** timer one-shot de 5000 ms reseteado con cada tecla.

```c
#define mainBACKLIGHT_TIMER_PERIOD pdMS_TO_TICKS(5000)

static BaseType_t xSimulatedBacklightOn = pdFALSE;
static TimerHandle_t xBacklightTimer;

/* Callback: se ejecuta si no se presionó ninguna tecla en 5 segundos */
static void prvBacklightTimerCallback(TimerHandle_t xTimer) {
    xSimulatedBacklightOn = pdFALSE;
    vPrintStringAndNumber("Timer expired, turning backlight OFF at time",
                          xTaskGetTickCount());
}

/* Tarea que detecta teclas y resetea el timer */
static void vKeyHitTask(void *pvParameters) {
    const TickType_t xShortDelay = pdMS_TO_TICKS(50);

    for (;;) {
        if (_kbhit() != 0) {
            TickType_t xTimeNow = xTaskGetTickCount();

            if (xSimulatedBacklightOn == pdFALSE) {
                /* Backlight estaba apagado: encender */
                xSimulatedBacklightOn = pdTRUE;
                vPrintStringAndNumber("Key pressed, turning backlight ON at time",
                                      xTimeNow);
            } else {
                /* Backlight ya encendido: resetear el timer */
                vPrintStringAndNumber("Key pressed, resetting software timer at time",
                                      xTimeNow);
            }

            /* xTimerReset inicia el timer si está Dormant,
               o reinicia el conteo si está Running */
            xTimerReset(xBacklightTimer, xShortDelay);
            (void) _getch();
        }
    }
}

int main(void) {
    xBacklightTimer = xTimerCreate("Backlight",
                                    mainBACKLIGHT_TIMER_PERIOD,
                                    pdFALSE,   /* one-shot */
                                    0,
                                    prvBacklightTimerCallback);
    xTaskCreate(vKeyHitTask, "KeyHit", configMINIMAL_STACK_SIZE, NULL, 1, NULL);
    vTaskStartScheduler();
    for (;;);
}
```

**Salida de ejemplo:**
```
Key pressed, turning backlight ON at time         812
Key pressed, resetting software timer at time    1813
Key pressed, resetting software timer at time    3114
Key pressed, resetting software timer at time    4015
Key pressed, resetting software timer at time    5016
Timer expired, turning backlight OFF at time    10016
```

El backlight se apaga exactamente 5000 ticks después del último reset (5016 + 5000 = 10016).

> **Adaptación para STM32:** en lugar de `_kbhit()` (Windows), usar una ISR de GPIO con `xTimerResetFromISR()`. Este patrón es muy común en interfaces de usuario con timeout de pantalla.


## 6.9 Resumen de APIs y Buenas Prácticas

### Tabla de APIs principales

| Función | Descripción | ¿Desde ISR? |
|---------|-------------|-------------|
| `xTimerCreate()` | Crea un timer (estado Dormant) | No |
| `xTimerStart()` | Inicia o reinicia un timer | No (usar `xTimerStartFromISR`) |
| `xTimerStop()` | Detiene un timer (→ Dormant) | No (usar `xTimerStopFromISR`) |
| `xTimerReset()` | Reinicia el conteo; inicia si estaba Dormant | No (usar `xTimerResetFromISR`) |
| `xTimerChangePeriod()` | Cambia el período; inicia si estaba Dormant | No (usar `...FromISR`) |
| `xTimerDelete()` | Elimina un timer | No |
| `vTimerSetTimerID()` | Establece el ID del timer | Sí (acceso directo) |
| `pvTimerGetTimerID()` | Lee el ID del timer | Sí (acceso directo) |
| `xTimerIsTimerActive()` | Verifica si el timer está en Running | Sí |
| `pcTimerGetName()` | Obtiene el nombre de texto del timer | Sí |

### Buenas prácticas

1. **Los callbacks deben ser cortos y no bloqueantes.** Si necesitas hacer algo largo, usa el callback para despertar una tarea dedicada (con un semáforo o cola).

2. **Desde un callback, usa `xTicksToWait = 0`** en cualquier API que escriba a la timer command queue. Nunca bloquear la daemon task.

3. **Configura `configTIMER_TASK_PRIORITY` adecuadamente.** Si la daemon task tiene muy baja prioridad, los callbacks se ejecutarán tarde (con jitter).

4. **Usa `pdMS_TO_TICKS()`** para especificar períodos en milisegundos en lugar de ticks crudos — el código es más portable si cambia `configTICK_RATE_HZ`.

5. **Para STM32:** resetear un timer desde una ISR de botón con `xTimerResetFromISR()` es el patrón estándar para interfaces con timeout (backlight, sleep mode, auto-apagado).

### Comparación: Software Timer vs Hardware Timer vs vTaskDelay

| Criterio | Software Timer | Hardware Timer | `vTaskDelay()` |
|----------|---------------|----------------|----------------|
| Recursos hardware | Ninguno | Timer periférico | Ninguno |
| Precisión | Resolución = 1 tick | Alta (nanosegundos) | Resolución = 1 tick |
| Cantidad simultánea | Ilimitada (heap) | Limitada por hardware | 1 por tarea |
| Callback en ISR | No (en tarea daemon) | Sí (en ISR) | N/A |
| Bloquea la tarea | No | No | Sí |
| Uso típico | Timeouts, periódico software | PWM, captura, alta precisión | Delays en tareas |
